In [ ]:
# ML Framework: Hugging Face Sentiment Monitor
**Name:** Manu Sri Mogalapalli 
**PRN:** 1302250186 
**Class:** S Y BCom FinTech | AI ML 1 - CCA2

## Shared setup (run first)

In [ ]:
!pip -q install transformers pandas openpyxl matplotlib

In [ ]:
import re, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline
warnings.filterwarnings("ignore")

sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

def classify(texts):
    """Run the model on a list of texts -> list of {'label','score'} (long texts are truncated)."""
    return sentiment(list(texts), truncation=True, batch_size=16)

def signed(r):
    """One number per text: +confidence if POSITIVE, -confidence if NEGATIVE (range -1 to +1)."""
    return r["score"] if r["label"] == "POSITIVE" else -r["score"]

## Problem 1: Sentiment Robustness Test Suite
**Problem statement.** The original notebook shows the model on a few easy sentences and notes that "the model is not always correct". Before a bank trusts it, the team needs a **systematic test suite**. Build labelled test cases in four categories (**negation, intensifiers, typos/slang, neutral questions**), measure the pass rate for each, and list every failure, so the team knows *where* the model cannot be trusted.

**Why it is original.** It turns "the model can be wrong" into **measured evidence by category**, like unit-testing for AI.

In [ ]:
tests = [
 # (text, expected label, category)
 ("The app is fast.",                              "POSITIVE", "Negation"),
 ("The app is not fast.",                          "NEGATIVE", "Negation"),
 ("The payment did not fail this time.",           "POSITIVE", "Negation"),
 ("I am not happy with the support.",              "NEGATIVE", "Negation"),
 ("Never had a problem with transfers.",           "POSITIVE", "Negation"),
 ("The service is good.",                          "POSITIVE", "Intensifier"),
 ("The service is extremely good.",                "POSITIVE", "Intensifier"),
 ("The charges are high.",                         "NEGATIVE", "Intensifier"),
 ("The charges are unbelievably high.",            "NEGATIVE", "Intensifier"),
 ("Great app, loved it!",                          "POSITIVE", "Typos/Slang"),
 ("Gr8 app, lovd it!!",                            "POSITIVE", "Typos/Slang"),
 ("worst app evr, keeps crashng",                  "NEGATIVE", "Typos/Slang"),
 ("app is lit, payments r super quick",            "POSITIVE", "Typos/Slang"),
 ("How do I reset my password?",                   "NEUTRAL",  "Neutral question"),
 ("What is the interest rate on savings accounts?","NEUTRAL",  "Neutral question"),
 ("Where is the nearest branch?",                  "NEUTRAL",  "Neutral question"),
]
df = pd.DataFrame(tests, columns=["Text", "Expected", "Category"])
res = classify(df["Text"])
df["Predicted"] = [r["label"] for r in res]
df["Confidence"] = [round(r["score"], 2) for r in res]

# The model has no NEUTRAL class, so for neutral questions the *fair* test is: does it stay unsure (confidence < 0.80)?
df["Pass"] = np.where(df.Expected == "NEUTRAL", df.Confidence < 0.80, df.Predicted == df.Expected)

report = df.groupby("Category")["Pass"].agg(Tests="count", Passed="sum")
report["Pass_Rate_%"] = (100 * report.Passed / report.Tests).round(0)
display(report)
print("Overall pass rate: %.0f%%" % (100 * df.Pass.mean()))
print("\nFAILURES (where the model cannot be trusted):")
display(df[~df.Pass][["Category", "Text", "Expected", "Predicted", "Confidence"]])
df.to_excel("p1_robustness_results.xlsx", index=False)

## Problem 2: Three-Way Sentiment (adding a "Neutral" class)
**Problem statement.** The model only says POSITIVE or NEGATIVE, so a factual headline like *"RBI keeps repo rate unchanged"* is forced into one of them, and the "overall mood" in the original is distorted. Create a **three-way classifier** (POSITIVE / NEUTRAL / NEGATIVE) by treating low-confidence predictions as NEUTRAL. Compare several confidence thresholds against **human labels** and pick the threshold that agrees with humans most often. Then compute a corrected **Market Mood Index**.

**Why it is original.** It fixes a *structural limitation* of the model (no neutral class) and tunes the fix with evidence rather than guessing.

In [ ]:
headlines = pd.DataFrame([
 ("Sensex surges 800 points as foreign investors return",        "POSITIVE"),
 ("Fintech startup raises record funding round",                 "POSITIVE"),
 ("Bank reports highest quarterly profit in five years",         "POSITIVE"),
 ("Banking stocks tumble after surprise rise in bad loans",      "NEGATIVE"),
 ("Inflation hits six-month high, consumers worry",              "NEGATIVE"),
 ("Rupee slides to record low against the dollar",               "NEGATIVE"),
 ("RBI keeps repo rate unchanged, markets react calmly",         "NEUTRAL"),
 ("RBI governor to address the media on Friday",                 "NEUTRAL"),
 ("Bank announces its annual general meeting date",              "NEUTRAL"),
 ("Quarterly results of three lenders due next week",            "NEUTRAL"),
], columns=["Headline", "Human_Label"])

res = classify(headlines.Headline)
headlines["Model_Label"] = [r["label"] for r in res]
headlines["Confidence"]  = [r["score"] for r in res]
headlines["Signed"]      = [signed(r) for r in res]

def three_way(label, conf, T):
    return label if conf >= T else "NEUTRAL"

rows = []
for T in [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]:
    pred = [three_way(l, c, T) for l, c in zip(headlines.Model_Label, headlines.Confidence)]
    rows.append({"Threshold": T, "Agreement_with_Humans_%": round(100 * np.mean(np.array(pred) == headlines.Human_Label), 0),
                 "Neutral_Share_%": round(100 * np.mean(np.array(pred) == "NEUTRAL"), 0)})
sweep = pd.DataFrame(rows)
display(sweep)

best_T = sweep.sort_values(["Agreement_with_Humans_%", "Threshold"], ascending=[False, True]).iloc[0]["Threshold"]
headlines["Three_Way"] = [three_way(l, c, best_T) for l, c in zip(headlines.Model_Label, headlines.Confidence)]
display(headlines[["Headline", "Human_Label", "Model_Label", "Confidence", "Three_Way"]].round(2))

pos = (headlines.Three_Way == "POSITIVE").sum(); neg = (headlines.Three_Way == "NEGATIVE").sum()
print(f"Best threshold = {best_T}")
print(f"Original-style mood (binary): {(headlines.Model_Label == 'POSITIVE').mean():.0%} positive")
print(f"Market Mood Index = (positive - negative) / total = {(pos - neg) / len(headlines):+.2f}   (range -1 to +1)")

## Problem 3: Aspect-Based Review Analyzer
**Problem statement.** One review can praise one feature and attack another: *"UPI is instant but the support team never replies."* Scoring the whole review hides this. Split every review into **sentences**, score each sentence, **tag it with the feature it mentions** (UPI, KYC, Support, Fees, Loans, App Performance), and produce a table that shows **which feature is hurting the bank most**.

**Why it is original.** The original gives one score per text. This gives **feature-level** insight a product manager can act on.

In [ ]:
reviews = [
 "UPI payments are instant. But the support team never replies to my emails.",
 "The KYC process was smooth and quick. The app keeps crashing during login though.",
 "Hidden charges on my card again. Very disappointed with the fees.",
 "Loan approval was fast and the interest rate is fair. Great experience overall.",
 "The app is slow and the UPI transfer failed twice.",
 "Customer care was polite and solved my issue in minutes.",
 "KYC verification keeps getting rejected for no reason. Support is useless.",
 "Love the new dashboard design. The loan EMI calculator is really helpful.",
 "Annual fee was charged without any notice, this is unacceptable.",
 "UPI is reliable and the app loads quickly.",
]

ASPECTS = {
    "UPI":             ["upi", "transfer", "payment"],
    "KYC":             ["kyc", "verification"],
    "Support":         ["support", "customer care", "reply", "replies", "care"],
    "Fees & Charges":  ["charge", "fee", "hidden"],
    "Loans":           ["loan", "emi", "interest"],
    "App Performance": ["app", "crash", "slow", "login", "loads", "dashboard"],
}

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def find_aspects(sentence):
    s = sentence.lower()
    found = [a for a, kws in ASPECTS.items() if any(k in s for k in kws)]
    return found or ["General"]

rows = [(i, s) for i, rv in enumerate(reviews) for s in split_sentences(rv)]
sent_df = pd.DataFrame(rows, columns=["Review_No", "Sentence"])
res = classify(sent_df.Sentence)
sent_df["Label"]  = [r["label"] for r in res]
sent_df["Signed"] = [signed(r) for r in res]
sent_df["Aspects"] = sent_df.Sentence.apply(find_aspects)

long = sent_df.explode("Aspects").rename(columns={"Aspects": "Aspect"})
aspect_table = long.groupby("Aspect").agg(Mentions=("Sentence", "count"),
                                           Avg_Sentiment=("Signed", "mean"),
                                           Negative_Share=("Label", lambda x: (x == "NEGATIVE").mean())).round(2)
aspect_table = aspect_table.sort_values("Avg_Sentiment")
display(aspect_table)
print("Biggest pain point :", aspect_table.index[0])
print("Biggest strength   :", aspect_table.index[-1])

with pd.ExcelWriter("p3_aspect_analysis.xlsx") as xw:
    long.to_excel(xw, sheet_name="Sentence_Level", index=False)
    aspect_table.to_excel(xw, sheet_name="Aspect_Summary")

aspect_table["Avg_Sentiment"].plot(kind="barh", color=["#d9534f" if v < 0 else "#5cb85c" for v in aspect_table["Avg_Sentiment"]],
                                   title="Average sentiment by feature", figsize=(6, 3.5))
plt.axvline(0, color="black", lw=0.8); plt.tight_layout(); plt.show()

## Problem 4: Sentiment Early-Warning Dashboard
**Problem statement.** A single batch summary (like the original Problem 5) cannot tell the bank *when* customers started getting unhappy. Simulate **8 weeks of app reviews** (a bad app update lands in week 6), compute the **weekly Net Sentiment Score**, learn the *normal range* from the first 4 weeks, and **raise an alert** whenever a week falls more than 2 standard deviations below normal. Plot the trend with the alert zone.

**Why it is original.** It converts one-off classification into **monitoring over time**, which is what the title "Sentiment *Monitor*" really implies.

In [ ]:
rng = np.random.default_rng(11)
good = ["Payments are quick and smooth.", "Love the clean new design.", "Support resolved my issue fast.",
        "Very easy to use, great app.", "Loan process was simple and fast.", "Reliable app, no problems at all."]
bad  = ["The app keeps crashing after the update.", "Payment failed and money was deducted.",
        "Terrible experience, login never works.", "Support never replies, very frustrating.",
        "Hidden charges again, very disappointed.", "The update made the app painfully slow."]

neg_prob = [0.20, 0.25, 0.20, 0.22, 0.30, 0.65, 0.70, 0.75]        # week 6 onwards: bad update
records = []
for w, p in enumerate(neg_prob, start=1):
    for _ in range(12):
        records.append({"Week": w, "Review": rng.choice(bad if rng.random() < p else good)})
reviews_ts = pd.DataFrame(records)

res = classify(reviews_ts.Review)
reviews_ts["Signed"] = [signed(r) for r in res]

weekly = reviews_ts.groupby("Week")["Signed"].agg(Net_Sentiment="mean", Reviews="count").round(3)
baseline = weekly.loc[1:4, "Net_Sentiment"]
mu, sd = baseline.mean(), max(baseline.std(), 0.05)          # floor on sd so a very calm baseline does not over-alert
weekly["Alert_Level"] = mu - 2 * sd
weekly["ALERT"] = np.where(weekly.Net_Sentiment < weekly.Alert_Level, "ALERT", "ok")
display(weekly.round(2))

first_alert = weekly.index[weekly.ALERT == "ALERT"]
print(f"Normal range learned from weeks 1-4: mean {mu:+.2f}, alert below {mu - 2*sd:+.2f}")
print("First alert in week:", int(first_alert[0]) if len(first_alert) else "none")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(weekly.index, weekly.Net_Sentiment, marker="o", label="Weekly net sentiment")
ax.axhline(mu - 2 * sd, color="red", ls="--", label="Alert line"); ax.axhline(0, color="grey", lw=0.6)
ax.scatter(first_alert, weekly.loc[first_alert, "Net_Sentiment"], color="red", zorder=5, label="Alert")
ax.set_xlabel("Week"); ax.set_ylabel("Net sentiment (-1 to +1)"); ax.legend(); plt.tight_layout()
plt.savefig("p4_sentiment_trend.png", dpi=120); plt.show()
weekly.to_excel("p4_weekly_sentiment.xlsx")

## Problem 5: Smart Complaint Priority Queue
**Problem statement.** The original escalates *every* strongly negative complaint equally. A real support team has limited staff and needs an **ordered queue**. Give each complaint a **0-100 priority score** built from: (a) how negative the model is, (b) **urgency keywords** (money deducted, blocked, fraud, no reply...), (c) **customer tier** (Premium customers get a bonus), and (d) **days the complaint has been open**. Output a ranked queue labelled P1 / P2 / P3.

**Why it is original.** It combines the **ML score with business rules**, which mirrors how traditional programming and ML are used together in practice.

In [ ]:
complaints = pd.DataFrame([
 (201, "Premium", 4, "My money was deducted twice and nobody is replying!"),
 (202, "Regular", 1, "The statement download is a bit slow."),
 (203, "Regular", 7, "My account was blocked and I cannot access my salary. This is fraud!"),
 (204, "Premium", 2, "Excellent service, my loan was approved quickly."),
 (205, "Regular", 3, "Card payment failed but the amount was debited."),
 (206, "Regular", 9, "No reply from support for a week. Very disappointed."),
 (207, "Premium", 1, "The new app layout is confusing."),
 (208, "Regular", 5, "Is the new interest rate applicable to old accounts?"),
], columns=["CustomerID", "Tier", "Days_Open", "Complaint"])

URGENT = ["deducted", "debited", "blocked", "fraud", "unauthorized", "no reply", "nobody", "cannot access", "twice", "scam"]

res = classify(complaints.Complaint)
complaints["Negativity"] = [r["score"] if r["label"] == "NEGATIVE" else 0.0 for r in res]    # 0 if positive
complaints["Urgent_Hits"] = complaints.Complaint.str.lower().apply(lambda t: sum(k in t for k in URGENT))

complaints["Priority_Score"] = (
      50 * complaints.Negativity                              # up to 50 pts: how upset
    + np.minimum(complaints.Urgent_Hits * 10, 30)             # up to 30 pts: money / access / fraud signals
    + np.where(complaints.Tier == "Premium", 10, 0)           # 10 pts: premium customer
    + np.minimum(complaints.Days_Open, 10)                    # up to 10 pts: waiting time
).round(0)
complaints["Queue"] = pd.cut(complaints.Priority_Score, [-1, 39, 69, 100], labels=["P3 - routine", "P2 - same day", "P1 - immediate"])

queue = complaints.sort_values("Priority_Score", ascending=False)[
    ["CustomerID", "Tier", "Days_Open", "Complaint", "Negativity", "Urgent_Hits", "Priority_Score", "Queue"]].round(2)
display(queue)
print(queue.Queue.value_counts().to_string())
queue.to_excel("p5_priority_queue.xlsx", index=False)

## Problem 6: Tone-Safe Reminder Rewriter (with fact preservation)
**Problem statement.** The original Problem 6 only *flags* harsh EMI/bill reminders for a human to rewrite. Automate the fix: for each draft, **extract the key facts** (amount and due date), generate candidate rewrites from approved polite templates, **score each candidate with the model**, and choose the most positive one that passes the safe-tone threshold. Every rewrite must **contain the original amount and due date**; if a draft has no facts to preserve, send it to a human instead of inventing them.

**Why it is original.** It closes the loop, **detect, fix, verify**, and adds a safety rule that the fix must not change the financial facts.

In [ ]:
drafts = [
 "Thank you for banking with us! Your EMI of Rs 4,500 is due on the 5th.",
 "You have failed to pay your EMI of Rs 4,500 due on the 5th. Pay immediately or face penalties.",
 "Your credit card bill of Rs 12,300 is overdue since the 10th and your account will be blocked.",
 "Your payment is overdue and your account will be blocked.",
]

TEMPLATES = [
 "Hi! A friendly reminder about your payment of {amount} (due date: {due}). Thank you for being a valued customer.",
 "Thank you for banking with us! Your payment of {amount} (due date: {due}) is waiting, and we are happy to help if you need anything.",
 "Good news: you can settle {amount} (due date: {due}) in a few taps in the app. Thanks for choosing us!",
]
SAFE_SCORE = 0.90
THREAT_WORDS = ["penalt", "blocked", "failed", "immediately", "overdue", "legal"]

def extract_facts(text):
    amount = re.search(r"Rs\.?\s?[\d,]+", text)
    due = re.search(r"\b(\d{1,2})(st|nd|rd|th)\b", text)
    return (amount.group(0) if amount else None), (due.group(0) if due else None)

orig = classify(drafts)
rows = []
for text, o in zip(drafts, orig):
    if o["label"] == "POSITIVE" and o["score"] >= SAFE_SCORE and not any(w in text.lower() for w in THREAT_WORDS):
        rows.append((text, "SEND AS IS", text, o["score"])); continue
    amount, due = extract_facts(text)
    if not amount or not due:
        rows.append((text, "MANUAL: amount/date missing, human must rewrite", "", o["score"])); continue
    candidates = [t.format(amount=amount, due=due) for t in TEMPLATES]
    scored = classify(candidates)
    ok = [(r["score"], c) for r, c in zip(scored, candidates)
          if r["label"] == "POSITIVE" and r["score"] >= SAFE_SCORE and amount in c and due in c]
    if ok:
        best = max(ok)
        rows.append((text, "AUTO-REWRITTEN", best[1], best[0]))
    else:
        rows.append((text, "MANUAL: no template passed the tone check", "", o["score"]))

out = pd.DataFrame(rows, columns=["Original_Draft", "Decision", "Final_Message", "Final_Tone_Score"]).round(2)
pd.set_option("display.max_colwidth", 120)
display(out)
print(out.Decision.value_counts().to_string())
out.to_excel("p6_reminder_rewrites.xlsx", index=False)